# Cadastral Map Boundary Segmentation: End-to-End U-Net Pipeline

This notebook provides a complete walk-through of the deep learning pipeline to segment land parcel boundaries on cadastral maps using PyTorch U-Net. It contains the code for hardware checking, data synthesis, the U-Net model architecture, a fast cached dataset loader, the training loop, tiled predictor, and visualization.

## 1. Hardware & Environment Check

We import the required libraries and inspect the host hardware. The code dynamically checks for NVIDIA GPUs (via `CUDA`) or AMD/Intel GPUs (via `DirectML`) on Windows before defaulting to CPU.

In [1]:
import sys
import os
import random
import re
import csv
import glob
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Device selection: CUDA -> DirectML -> CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Device Status: NVIDIA GPU Enabled via CUDA ({torch.cuda.get_device_name(0)})")
else:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Device Status: Intel Arc / AMD GPU Enabled via Microsoft DirectML")
    except ImportError:
        device = torch.device("cpu")
        print("Device Status: CPU Mode (CUDA and torch-directml not installed)")

Device Status: CPU Mode (CUDA and torch-directml not installed)


## 2. Dataset Synthesis (Map Degradation)

To train the model to segment boundary lines under dirty scan conditions, we synthesize training data by degrading clean digital map images (dilations, blurring, uneven lighting, and gaussian noise).

In [2]:
def degrade_image(img_gray: np.ndarray, seed: int = 42) -> np.ndarray:
    """
    Applies synthetic scanning and aging artifacts to a clean digital map.
    """
    rng = np.random.default_rng(seed)
    h, w = img_gray.shape
    
    # 1. Start with clean image
    work = img_gray.copy()
    
    # 2. Add structural distortions (simulate bleeding ink or hand-drawn variance)
    k_size = rng.choice([3, 5])
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k_size, k_size))
    if rng.random() > 0.5:
        work = cv2.erode(work, kernel, iterations=1)
    else:
        work = cv2.dilate(work, kernel, iterations=1)
        
    # 3. Add Gaussian noise
    noise = rng.normal(0, rng.uniform(5, 15), (h, w)).astype(np.float32)
    work = np.clip(work.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    
    # 4. Add uneven illumination gradient (simulating page shadows)
    X, Y = np.meshgrid(np.linspace(0, 1, w), np.linspace(0, 1, h))
    grad = (X * rng.uniform(-40, 40)) + (Y * rng.uniform(-40, 40))
    work = np.clip(work.astype(np.float32) + grad, 0, 255).astype(np.uint8)
    
    # 5. Low-contrast blur
    work = cv2.GaussianBlur(work, (3, 3), 0)
    
    return work

## 3. PyTorch U-Net Architecture

We define a lightweight U-Net model architecture optimized to run quickly on standard consumer CPUs and GPUs.

In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=1, n_classes=1, bilinear=True):
        super().__init__()
        self.inc = DoubleConv(n_channels, 16)
        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        self.down3 = Down(64, 128)
        factor = 2 if bilinear else 1
        self.down4 = Down(128, 256 // factor)
        self.up1 = Up(256, 128 // factor, bilinear)
        self.up2 = Up(128, 64 // factor, bilinear)
        self.up3 = Up(64, 32 // factor, bilinear)
        self.up4 = Up(32, 16, bilinear)
        self.outc = nn.Conv2d(16, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

## 4. Memory-Cached PyTorch Dataset

We implement the Dataset loading logic. In order to optimize training performance, we load all unique images into RAM during initialization, preventing redundant disk-decoding overhead during training iterations.

In [4]:
class CadastralDataset(Dataset):
    def __init__(self, manifest_csv, patch_size=256, tiles_per_image=30):
        self.patch_size = patch_size
        self.tiles_per_image = tiles_per_image
        self.pairs = []
        self.cached_pairs = []

        manifest_csv = Path(manifest_csv)
        if not manifest_csv.exists():
            raise FileNotFoundError(f"Manifest not found: {manifest_csv}")

        def load_gray(path):
            img = cv2.imread(str(path))
            if img is None:
                raise FileNotFoundError(f"Image not found: {path}")
            if img.ndim == 3:
                return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            return img

        # Read the splits manifest
        with open(manifest_csv, encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                img_path = Path(row["image_path"])
                if not img_path.is_absolute():
                    proj_root = Path(".").resolve()
                    img_path = proj_root / img_path

                clean_path = img_path
                if "_deg_" in img_path.name:
                    clean_name = re.sub(r"_deg_\d+", "_clean", img_path.name)
                    clean_path = img_path.parent / clean_name

                if img_path.exists() and clean_path.exists():
                    self.pairs.append({"input_path": img_path, "clean_path": clean_path})

        print(f"Loading and caching {len(self.pairs)} image pairs from {manifest_csv.name} in memory...")
        for pair in self.pairs:
            input_gray = load_gray(pair["input_path"])
            clean_gray = load_gray(pair["clean_path"])
            target_mask = (clean_gray < 220).astype(np.float32)

            self.cached_pairs.append({
                "input_gray": input_gray,
                "target_mask": target_mask
            })

    def __len__(self) -> int:
        return len(self.cached_pairs) * self.tiles_per_image

    def __getitem__(self, idx: int):
        pair_idx = idx // self.tiles_per_image
        pair = self.cached_pairs[pair_idx]
        input_gray = pair["input_gray"]
        target_mask = pair["target_mask"]

        h, w = input_gray.shape
        p = self.patch_size
        best_x = random.randint(0, max(0, w - p))
        best_y = random.randint(0, max(0, h - p))

        # Try up to 8 times to find an active boundary patch
        for _ in range(8):
            tx = random.randint(0, max(0, w - p))
            ty = random.randint(0, max(0, h - p))
            if target_mask[ty:ty + p, tx:tx + p].mean() > 0.015:
                best_x, best_y = tx, ty
                break

        input_patch = input_gray[best_y:best_y + p, best_x:best_x + p]
        target_patch = target_mask[best_y:best_y + p, best_x:best_x + p]

        # Pad if tile is smaller than patch size
        if input_patch.shape[0] < p or input_patch.shape[1] < p:
            input_patch = cv2.copyMakeBorder(input_patch, 0, p - input_patch.shape[0], 0, p - input_patch.shape[1], cv2.BORDER_CONSTANT, value=255)
            target_patch = cv2.copyMakeBorder(target_patch, 0, p - target_patch.shape[0], 0, p - target_patch.shape[1], cv2.BORDER_CONSTANT, value=0)

        # Data augmentation
        if random.random() > 0.5:
            input_patch = np.fliplr(input_patch)
            target_patch = np.fliplr(target_patch)
        if random.random() > 0.5:
            input_patch = np.flipud(input_patch)
            target_patch = np.flipud(target_patch)

        input_tensor = torch.from_numpy(input_patch.copy()).float().unsqueeze(0) / 255.0
        input_tensor = 1.0 - input_tensor  # Invert so lines are 1.0
        target_tensor = torch.from_numpy(target_patch.copy()).float().unsqueeze(0)

        return input_tensor, target_tensor

## 5. DiceBCELoss & Training Setup

We define a custom loss function combining Binary Cross-Entropy and Dice Loss to counter the high class imbalance of boundary lines.

In [5]:
class DiceBCELoss(nn.Module):
    def __init__(self, weight_bce=0.5):
        super().__init__()
        self.weight_bce = weight_bce
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1.0):
        bce_loss = self.bce(inputs, targets)
        inputs_prob = torch.sigmoid(inputs)
        inputs_flat = inputs_prob.view(-1)
        targets_flat = targets.view(-1)
        intersection = (inputs_flat * targets_flat).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs_flat.sum() + targets_flat.sum() + smooth)
        return self.weight_bce * bce_loss + (1 - self.weight_bce) * dice_loss

def run_training(epochs=25, batch_size=8):
    train_manifest = Path("data/synthetic/train.csv")
    val_manifest = Path("data/synthetic/val.csv")
    output_weights = Path("data/unet_weights.pth")

    if not train_manifest.exists() or not val_manifest.exists():
        print("Error: Run synthesize_dataset.py first to split raw data.")
        return

    train_dataset = CadastralDataset(train_manifest, patch_size=256, tiles_per_image=30)
    val_dataset = CadastralDataset(val_manifest, patch_size=256, tiles_per_image=15)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = UNet(n_channels=1, n_classes=1, bilinear=True).to(device)
    criterion = DiceBCELoss(weight_bce=0.5)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=2, factor=0.5)

    best_val_loss = float("inf")
    print(f"Starting training loop on device: {device}...")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        epoch_train_loss = train_loss / len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)

        epoch_val_loss = val_loss / len(val_loader.dataset)
        scheduler.step(epoch_val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | LR: {current_lr}")
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), str(output_weights))
            print(f"  -> Checkpoint saved to {output_weights}")

## 6. Tiled Prediction Model

Since full maps are extremely large, we perform sliding window inference with configurable overlapping margins to avoid edge artifacts, then average the raw probabilities.

In [6]:
class UNetPredictor:
    def __init__(self, weights_path, patch_size=256, overlap=64):
        self.patch_size = patch_size
        self.overlap = overlap
        self.device = device
        self.model = UNet(n_channels=1, n_classes=1, bilinear=True)
        self.model.load_state_dict(torch.load(str(weights_path), map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    @torch.no_grad()
    def predict_mask(self, gray, threshold=0.5) -> np.ndarray:
        h, w = gray.shape[:2]
        p = self.patch_size
        step = p - self.overlap

        accum = np.zeros((h, w), dtype=np.float32)
        count = np.zeros((h, w), dtype=np.float32)

        y_coords = list(range(0, max(1, h - p + 1), step))
        if y_coords[-1] + p < h: y_coords.append(h - p)
        x_coords = list(range(0, max(1, w - p + 1), step))
        if x_coords[-1] + p < w: x_coords.append(w - p)

        for y in y_coords:
            for x in x_coords:
                crop = gray[y:y + p, x:x + p]
                ch, cw = crop.shape[:2]
                padded_crop = cv2.copyMakeBorder(crop, 0, p - ch, 0, p - cw, cv2.BORDER_CONSTANT, value=255) if (ch < p or cw < p) else crop

                tile_tensor = torch.from_numpy(padded_crop).float().unsqueeze(0).unsqueeze(0) / 255.0
                tile_tensor = 1.0 - tile_tensor
                tile_tensor = tile_tensor.to(self.device)

                logits = self.model(tile_tensor)
                probs = torch.sigmoid(logits).squeeze(0).squeeze(0).cpu().numpy()
                prob_patch = probs[:ch, :cw]

                accum[y:y + ch, x:x + cw] += prob_patch
                count[y:y + ch, x:x + cw] += 1.0

        final_probs = np.divide(accum, count, out=np.zeros_like(accum), where=count > 0)
        return (final_probs >= threshold).astype(np.uint8) * 255

## 7. Visual Boundary Segmentation Demonstration

We load the newly trained 25-epoch weights and evaluate them on one of the standardized map PNGs.

In [7]:
weights_path = Path("data/unet_weights.pth")
standardized_images = glob.glob("data/standardized/*.png")

# Select village map images specifically
village_images = [img for img in standardized_images if "Image_to_PDF" not in Path(img).name]
if not village_images:
    village_images = standardized_images

if village_images and weights_path.exists():
    img_path = Path(village_images[0])
    img = cv2.imread(str(img_path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img

    # Run prediction
    predictor = UNetPredictor(weights_path)
    pred_mask = predictor.predict_mask(gray, threshold=0.5)

    # Plot side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(gray, cmap="gray")
    axes[0].set_title(f"Original Map ({img_path.name})")
    axes[0].axis("off")

    # Display boundaries as black lines on a white canvas
    axes[1].imshow(255 - pred_mask, cmap="gray")
    axes[1].set_title("U-Net Segmented Boundaries")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Error: Run training and synthesize_dataset.py first!")

AttributeError: 'NoneType' object has no attribute 'ndim'

## 8. Custom Input Inference (Images, PDFs, and Word DOC/DOCX)

Run the cell below to load your own custom input file. It supports standard images (PNG, JPG, TIFF), multi-page PDFs, and Word documents (`.doc` and `.docx` containing embedded map images).

In [ ]:
# Change this path to point to your custom file! (e.g., png, jpg, pdf, docx, doc)
custom_file_path = "data/standardized/05_अकोला_01_तेल्हारा_270500010045860000_दिवाणझरी.png"

custom_path = Path(custom_file_path)
if custom_path.exists() and weights_path.exists():
    # Import the pipeline's source loader
    from parcel_extraction.input_normalizer import load_source
    
    try:
        print(f"Loading source file: {custom_path}...")
        sources = load_source(custom_path)
        print(f"Found {len(sources)} page(s)/image(s) to process.")
    except Exception as e:
        print(f"Error loading custom input: {e}")
        sources = []

    for idx, (name, img) in enumerate(sources, 1):
        print(f"[{idx}/{len(sources)}] Segmenting {name}...")
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img

        # Predict custom boundaries using U-Net
        predictor = UNetPredictor(weights_path)
        pred_mask = predictor.predict_mask(gray, threshold=0.5)

        # Plot side-by-side
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        axes[0].imshow(gray, cmap="gray")
        axes[0].set_title(f"Original Source ({name})")
        axes[0].axis("off")

        # Invert boundaries so they are black lines on a white canvas
        axes[1].imshow(255 - pred_mask, cmap="gray")
        axes[1].set_title("U-Net Segmented Boundaries")
        axes[1].axis("off")

        plt.tight_layout()
        plt.show()

        # Save segmented boundaries
        out_stem = Path(name).stem
        out_name = custom_path.parent / f"{out_stem}_boundaries.png"
        cv2.imwrite(str(out_name), 255 - pred_mask)
        print(f"  -> Segmented boundary mask saved to: {out_name}")
else:
    if not custom_path.exists():
        print(f"Error: File not found at '{custom_file_path}'")
    if not weights_path.exists():
        print(f"Error: Trained weights not found at '{weights_path}'")